# SenGuard-MSC kernel E: amendment 1 (extended cohorts, head-to-head, robustness, drift clock)
Needs kernel A attached. Frozen weights are verified before use.

In [ ]:
# Restores the frozen outputs of kernel A at their original absolute paths, so hash checks stay meaningful.
import shutil
from pathlib import Path
hits = sorted(Path('/kaggle/input').rglob('frozen_model.json'))
if not hits:
    raise FileNotFoundError('Kernel A output is not attached: no frozen_model.json under /kaggle/input.')
src = hits[0].parents[1]
dst = Path('/kaggle/working/senguard_msc')
if not dst.exists():
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.git', '.venv_check'))
print('Restored', src, '->', dst, '|', sorted(p.name for p in dst.iterdir()))

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], capture_output=True, text=True).returncode)

# SenGuard-MSC · Notebook 06 — Protocol amendment: more MSC cohorts, head-to-head, robustness, drift clock

Written after G3 was seen, so this is an amendment and is labelled as one. Cohorts below were chosen from metadata before any of them was scored. The amendment is written to disk and hashed before any expression file is read.

In [ ]:
import gc, gzip, io, json, os, re, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests
ROOT_GUESS = Path('/kaggle/working/senguard_msc') if Path('/kaggle/working/senguard_msc').exists() else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, EXT, DEV, ASC, BM
from scipy import stats
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
RAWX = ROOT / 'data/raw_extended'
RAWX.mkdir(parents=True, exist_ok=True)
FREEZE = json.loads((RESULTS / 'frozen_model.json').read_text())
for name, art in FREEZE['artifacts'].items():
    if sg.sha256(art['path']) != art['sha256']:
        raise RuntimeError(f'{name} changed after the freeze')
print('Frozen artifacts verified. Model weights and published SenCat weights are unchanged.')

AMENDMENT = """# Analysis protocol amendment 1

Fixed on 2026-11-01, after G3 results on GSE309666 and GSE178514 were seen, and before any cohort below was scored.
Inclusion came from GEO metadata only. Published and refit weights stay frozen.

## New cohorts
| Cohort | Tissue | Design | Use |
|---|---|---|---|
| GSE137186 | BM | 4 donors, P1, P5, P10, P15, 2 resequenced samples | Pooled; test-retest; authors report retained immunoregulatory potency |
| GSE275980 | MSC, tissue unstated in the abstract | 3 donors, P2, P6, P10 | Pooled |
| GSE125632 | BM | 5 donors, P2 vs P11 labelled senescent by authors | Pooled; labelled check |
| GSE194298 | BM | 4 donors, p1 vs p4/5, FBS and hPL | Pooled per donor; medium effect |
| GSE119987 | UC | 4 donors to replicative senescence, array, doubling level known | Pooled; early change |
| GSE56362 | BM | Donors at P3, P5, P7 with technical replicates, array | Pooled; test-retest |
| GSE233634 | UC | P3 to P12 in triplicate, no donor stated, counts as one unit | Pooled as one lineage |
| GSE269586 | WJ | GMP production, master bank to P2 or P3 | Not pooled: short-expansion comparison |
| GSE48662 | BM | Irradiated vs control; GVHD responders G1 (5) vs non-responders G2 (3), array | Labelled check; clinical-response question |

## Primary question
Across donors in new pooled cohorts, does the published SenCat score rise with passage? Test: two-sided exact binomial on donors, plus a cohort-level sign-flip test. Log1p weighted score on RNA-seq; rank-weighted form on arrays. Sensitivities: rank form everywhere, weights refit without BMMSC and preadipocytes, excluding GSE56362 (gene-normalized values), and excluding GSE233634.

## Head-to-head
Published SenCat, hUSI, SenMayo, the 7-gene signature of Yang 2023, the 40 AD-MSC biomarkers, Hallmark E2F and G2M (inverted), and CDKN2A, CDKN1A, IL6, GDF15, LMNB1 (inverted) transcripts, all in rank form. Reported: pooled donors rising, median within-donor rho, cohorts with a positive mean. Results on a score's own training cohort are flagged and left out of its pooled count. No single winner is declared unless its lead is clear of the uncertainty.

## Clinical-response question
One directional hypothesis: published SenCat rank score is higher in GVHD non-responders (G2) than responders (G1). Exact permutation, smallest one-sided p 1/56. The outcome will be reported either way, and it would be hypothesis-generating at any p.

## Labelled MSC checks
AUROC is allowed only where the authors define the classes: GSE48662 control vs irradiated, GSE125632 P2 vs P11 senescent. Within-cohort only, with no transported threshold.

## Drift clock (trained model)
Ridge regression of within-donor log2(passage + 1) on within-sample gene ranks, with leave-one-cohort-out validation and all selection inside training cohorts. It estimates passage-like drift and is not a senescence or potency measure. Negative control: passages shuffled within donors in training. It enters the demo only if held-out median within-donor rho is at least 0.8, at least 90% of held-out donors rise, and the shuffled control's median rho stays below 0.3.

## Robustness
Binomial thinning of counts, random gene dropout, technical test-retest, leave-one-sample-out normalization, medium effect, and absolute-score comparability across cohorts. These set guard thresholds in the tool from measurement instead of judgement.
"""
amend_path = REPORTS / 'ANALYSIS_AMENDMENT_1.md'
amend_path.write_text(AMENDMENT, encoding='utf-8')
print('Amendment sha256', sg.sha256(amend_path))

## Download and parse each cohort
Each cohort loads inside its own try block. A failure is recorded and doesn't stop the others.

In [ ]:
UA = {'User-Agent': 'SenGuard-MSC-audit/1.0'}
GEO = 'https://ftp.ncbi.nlm.nih.gov/geo'
CGI = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi'

def fetch(url, name=None):
    dest = RAWX / (name or url.rsplit('/', 1)[-1])
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    for attempt in range(5):
        try:
            if attempt < 3:
                with requests.get(url, headers=UA, stream=True, timeout=300) as r:
                    r.raise_for_status()
                    with open(dest, 'wb') as f:
                        for c in r.iter_content(1 << 20):
                            f.write(c)
            else:
                subprocess.run(['curl', '-fsSL', '--http1.1', '-A', UA['User-Agent'], '-o', str(dest), url], check=True, timeout=600)
            if dest.stat().st_size > 0:
                if dest.suffix == '.gz':
                    with gzip.open(dest, 'rb') as f:
                        while f.read(1 << 20):
                            pass
                return dest
        except Exception as e:
            print('  retry', dest.name, type(e).__name__, str(e)[:90])
        time.sleep(4 * (attempt + 1))
    raise RuntimeError(f'Could not download {url}')

def sdir(acc):
    return f'{GEO}/series/{acc[:-3]}nnn/{acc}'

def parse_matrix(path):
    meta, chars = {}, []
    rows = []
    in_table = False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!series_matrix_table_begin'):
                in_table = True
            elif line.startswith('!series_matrix_table_end'):
                break
            elif in_table:
                rows.append(line.rstrip('\n').replace('"', '').split('\t'))
            elif line.startswith('!Sample_'):
                parts = line.rstrip('\n').replace('"', '').split('\t')
                if parts[0] == '!Sample_characteristics_ch1':
                    chars.append(parts[1:])
                else:
                    meta[parts[0][8:]] = parts[1:]
    expr = pd.DataFrame(rows[1:], columns=rows[0]).set_index('ID_REF').apply(pd.to_numeric, errors='coerce').T
    m = pd.DataFrame({'sample_id': meta['geo_accession'], 'title': meta['title']})
    for col in chars:
        key = col[0].split(':')[0].strip().lower()
        m[key] = [c.split(':', 1)[1].strip() if ':' in c else c for c in col]
    return expr, m.set_index('sample_id')

def probe_symbols(gpl):
    annot = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/annot/{gpl}.annot.gz'
    soft = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/soft/{gpl}_family.soft.gz'
    try:
        path = fetch(annot, f'{gpl}.annot.gz')
    except Exception:
        path = fetch(soft, f'{gpl}_family.soft.gz')
    rows, header, on = [], None, False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!platform_table_begin'):
                on = True
                continue
            if line.startswith('!platform_table_end'):
                break
            if on:
                parts = line.rstrip('\n').split('\t')
                if header is None:
                    header = parts
                elif len(parts) == len(header):
                    rows.append(parts)
    t = pd.DataFrame(rows, columns=header)
    col = next((c for c in t.columns if c.lower() in ('gene symbol', 'gene_symbol', 'genesymbol', 'symbol', 'gene_name')), None)
    if col is None and 'gene_assignment' in t.columns:
        t['sym'] = t.gene_assignment.str.split('//').str[1].str.strip()
        col = 'sym'
    if col is None:
        raise RuntimeError(f'{gpl}: no symbol column in {list(t.columns)[:15]}')
    s = t.set_index('ID')[col].astype(str).str.strip()
    s = s[(s != '') & (s != 'nan') & (s != '---') & ~s.str.contains('///')]
    return s

ens2sym_tbl = pd.read_csv(PROC / f'{ASC}_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
sym2ens_tbl = pd.Series(ens2sym_tbl.index, index=ens2sym_tbl.values)
sym2ens_tbl = sym2ens_tbl[~sym2ens_tbl.index.duplicated(keep=False)]

def to_sym(expr):
    """Samples x genes to samples x symbols; Ensembl ids are mapped through annotation, keeping the highest-mean gene per symbol."""
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        c = cols.str.replace(r'\.\d+$', '', regex=True)
        e = expr.copy(); e.columns = c
        return sg.to_symbols(e.loc[:, ~e.columns.duplicated()], pd.DataFrame({'symbol': ens2sym_tbl.reindex(c.unique())}, index=c.unique()))
    e = expr.loc[:, ~pd.Index(cols).duplicated()]
    return e

def to_ens(expr):
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        e = expr.copy(); e.columns = cols.str.replace(r'\.\d+$', '', regex=True)
        return e.loc[:, ~e.columns.duplicated()]
    m = sym2ens_tbl.reindex(cols)
    e = expr.loc[:, m.notna().to_numpy()].copy()
    e.columns = m.dropna().to_numpy()
    return e.loc[:, ~e.columns.duplicated()]

COHORTS, LOAD_LOG = {}, []
def add(key, expr, meta, kind, tissue, pooled, role, note=''):
    """kind: counts, normcounts, fpkm, log_array."""
    meta = meta.loc[expr.index].copy()
    if not {'subject', 'passage', 'unit'} <= set(meta.columns):
        raise ValueError(f'{key}: meta needs subject, passage, unit; has {list(meta.columns)}')
    meta['passage'] = meta.passage.astype(float)
    COHORTS[key] = dict(expr=expr, meta=meta, kind=kind, tissue=tissue, pooled=pooled, role=role, note=note)
    LOAD_LOG.append({'cohort': key, 'status': 'loaded', 'samples': len(expr), 'genes': expr.shape[1], 'subjects': meta.subject.nunique(), 'units': meta.unit.nunique(), 'kind': kind, 'tissue': tissue, 'pooled': pooled, 'note': note})
    print(f'{key}: {expr.shape} subjects {meta.subject.nunique()} units {meta.unit.nunique()} passages {sorted(meta.passage.unique())[:10]} kind {kind}')

def attempt(key, fn):
    try:
        fn()
    except Exception as e:
        LOAD_LOG.append({'cohort': key, 'status': f'failed: {type(e).__name__}: {str(e)[:220]}'})
        print(key, 'FAILED', type(e).__name__, str(e)[:300])

def geo_samples(acc):
    t = requests.get(CGI, params={'acc': acc, 'targ': 'gsm', 'form': 'text', 'view': 'full'}, headers=UA, timeout=180).text
    rows = []
    for b in re.split(r'\^SAMPLE = ', t)[1:]:
        ch = {c.split(':')[0].strip().lower(): c.split(':', 1)[1].strip() for c in re.findall(r'!Sample_characteristics_ch1 = (.*)', b) if ':' in c}
        rows.append({'gsm': b.split('\n')[0].strip(), 'title': re.findall(r'!Sample_title = (.*)', b)[0].strip(), 'description': ' | '.join(x.strip() for x in re.findall(r'!Sample_description = (.*)', b)), **ch})
    time.sleep(0.5)
    return pd.DataFrame(rows)

# Already audited cohorts
def _asc():
    m = sg.load_manifest(); a = m[(m.study_accession == ASC)]
    e = pd.read_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    meta = pd.DataFrame({'subject': a.donor.astype(str), 'unit': a.donor.astype(str), 'passage': a.passage}, index=a.index)
    add(ASC, e, meta, 'normalized', 'AD', False, 'seen')
attempt(ASC, _asc)
def _bm():
    m = sg.load_manifest(); a = m[(m.study_accession == BM) & m.use_in_mrna_analysis]
    e = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    ids = a.candidate_biological_id.astype(int).astype(str)
    add(BM, e, pd.DataFrame({'subject': ids, 'unit': ids, 'passage': a.passage}, index=a.index), 'fpkm', 'BM', False, 'seen', 'Donors inferred')
attempt(BM, _bm)

def _137():
    p = fetch(f'{sdir("GSE137186")}/suppl/GSE137186_raw.counts.txt.gz')
    t = pd.read_csv(p, sep='\t', header=None, dtype=str, low_memory=False)
    head = t.iloc[:4]
    body = t.iloc[4:]
    body = body[~body[1].astype(str).str.startswith('__')]
    expr = body.set_index(1).iloc[:, 1:].apply(pd.to_numeric, errors='coerce').T
    names = [f'S{i}' for i in head.iloc[1, 2:]]
    expr.index = names
    meta = pd.DataFrame({'subject': head.iloc[2, 2:].to_numpy(), 'unit': head.iloc[2, 2:].to_numpy(), 'passage': head.iloc[3, 2:].astype(float).to_numpy()}, index=names)
    expr = expr.loc[:, ~expr.columns.duplicated()]
    add('GSE137186', expr, meta, 'counts', 'BM', True, 'new', 'Authors report retained immunoregulatory potency')
attempt('GSE137186', _137)

def _275():
    p = fetch(f'{sdir("GSE275980")}/suppl/GSE275980_raw_count.xlsx')
    t = pd.read_excel(p)
    t = t.drop_duplicates(t.columns[0]).set_index(t.columns[0])
    t.columns = [c.replace('P2C2', 'P2-C2') for c in t.columns]
    expr = t.apply(pd.to_numeric, errors='coerce').fillna(0).T
    parts = pd.Series(expr.index).str.extract(r'P(\d+)-C(\d)')
    meta = pd.DataFrame({'subject': ('C' + parts[1]).to_numpy(), 'unit': ('C' + parts[1]).to_numpy(), 'passage': parts[0].astype(float).to_numpy()}, index=expr.index)
    add('GSE275980', expr, meta, 'counts', 'MSC', True, 'new', 'Gene symbols only; C1-C3 are the three donors in the series title')
attempt('GSE275980', _275)

def _125():
    p = fetch(f'{sdir("GSE125632")}/suppl/GSE125632_FPKM_iMSC_manuscript.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False, on_bad_lines='skip', dtype=str)
    t = t[t.EnsemblID.astype(str).str.startswith('ENSG')].drop_duplicates('EnsemblID').set_index('EnsemblID')
    cols = [f'Sample_{i}' for i in range(1, 21)]
    expr = t[cols].apply(pd.to_numeric, errors='coerce').fillna(0).T
    g = geo_samples('GSE125632')
    meta = pd.DataFrame({'title': g.title.to_numpy(), 'subject': g.donor.to_numpy(), 'passage': g.passage.str.replace('P', '').astype(float).to_numpy()}, index=cols)
    meta['arm'] = np.where(meta.title.str.startswith('iMSC'), 'iMSC', 'MSC')
    meta['gender'] = g.gender.to_numpy()
    y = t.reindex(['ENSG00000129824'])[cols].apply(pd.to_numeric, errors='coerce').iloc[0]
    agree = float(((y > 1).to_numpy() == (meta.gender == 'Male').to_numpy()).mean())
    print('  GSE125632 column order check: RPS4Y1 agrees with GEO sex in', f'{agree:.0%}', 'of samples')
    keep = meta.arm == 'MSC'
    m = meta[keep].assign(unit=lambda d: d.subject)
    add('GSE125632', expr.loc[keep], m, 'fpkm', 'BM', True, 'new', f'Column order to GSM inferred; sex check agreement {agree:.0%}')
    im = meta[~keep].assign(unit=lambda d: d.subject)
    add('GSE125632_iMSC', expr.loc[~keep], im, 'fpkm', 'iPSC-derived', False, 'new', 'Reported separately')
attempt('GSE125632', _125)

def _194():
    p = fetch(f'{sdir("GSE194298")}/suppl/GSE194298_Norm_counts_RNAseq_hPL_vs_FBS.csv.gz')
    t = pd.read_csv(p, index_col=0)
    t.columns = [re.search(r'X(\d+)', c).group(1) for c in t.columns]
    g = geo_samples('GSE194298')
    g['num'] = g.title.str.extract(r'_(\d+)$')
    g = g.set_index('num').loc[t.columns]
    expr = t.T
    medium = g.treatment.str.strip().to_numpy()
    meta = pd.DataFrame({'donor': g.donor.to_numpy(), 'medium': medium, 'passage': g.passage.map({'p1': 1.0, 'p4/p5': 4.5}).to_numpy()}, index=t.columns)
    meta['subject'] = meta.donor + '_' + meta.medium
    meta['unit'] = meta.donor
    add('GSE194298', expr, meta, 'normcounts', 'BM', True, 'new', 'Medium arms averaged per donor; mBG arm has no p1')
attempt('GSE194298', _194)

def _233():
    p = fetch(f'{sdir("GSE233634")}/suppl/GSE233634_gene_FPKM.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('geneID')
    plane = [c for c in t.columns if re.fullmatch(r'P\d+_\d', c)]
    bio = [c for c in t.columns if re.fullmatch(r'BioR\d_\d', c)]
    expr = t[plane].apply(pd.to_numeric, errors='coerce').T
    meta = pd.DataFrame({'passage': pd.Series(plane).str.extract(r'P(\d+)_')[0].astype(float).to_numpy(), 'subject': 'lineage', 'unit': 'lineage', 'rep': pd.Series(plane).str.extract(r'_(\d)')[0].to_numpy()}, index=plane)
    add('GSE233634', expr, meta, 'fpkm', 'UC', True, 'new', 'No donor stated; one unit')
    COHORTS['GSE233634']['bioreactor'] = t[bio].apply(pd.to_numeric, errors='coerce').T
attempt('GSE233634', _233)

def _269():
    p = fetch(f'{sdir("GSE269586")}/suppl/GSE269586_all_counts_merged.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('id')
    g = geo_samples('GSE269586')
    g = g[g.title.str.contains('RNAseq')].copy()
    g['col'] = 'raw_' + g.description.str.split(' \\| ').str[0]
    missing = [c for c in g.col if c not in t.columns]
    if missing:
        raise RuntimeError(f'columns missing: {missing[:4]}')
    expr = t[g.col.tolist()].apply(pd.to_numeric, errors='coerce').fillna(0).T
    expr.index = g.gsm.to_numpy()
    donor = g.donor.str.replace('donor ', '').to_numpy()
    meta = pd.DataFrame({'subject': donor, 'unit': donor, 'passage': g.passage.astype(float).to_numpy()}, index=g.gsm.to_numpy())
    add('GSE269586', expr, meta, 'counts', 'WJ', False, 'new', 'GMP production; short expansion; not pooled')
attempt('GSE269586', _269)

def _119():
    p = fetch(f'{sdir("GSE119987")}/matrix/GSE119987_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL571')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    meta = pd.DataFrame({'subject': m['donor population name'], 'unit': m['donor population name'].str.replace(r'\.\d+$', '', regex=True), 'passage': m['passage number'].astype(float), 'pdl': pd.to_numeric(m.get('population doubling level'), errors='coerce'), 'batch': m.get('overall batch')}, index=m.index)
    add('GSE119987', e, meta, 'log_array', 'UC', True, 'new', 'Cultured to replicative senescence')
attempt('GSE119987', _119)

def _563():
    p = fetch(f'{sdir("GSE56362")}/matrix/GSE56362_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL18503')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    has = m['passage'].astype(str).str.contains(r'\d')
    meta = pd.DataFrame({'subject': m['donor id'], 'unit': m['donor id'], 'passage': m['passage'].astype(str).str.extract(r'(\d+)')[0].astype(float)}, index=m.index)[has]
    e = e.loc[meta.index]
    add('GSE56362', e, meta, 'log_array', 'BM', True, 'new', 'Values look gene-normalized; technical triplicates averaged')
attempt('GSE56362', _563)

def _486():
    p = fetch(f'{sdir("GSE48662")}/matrix/GSE48662-GPL13497_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL13497')
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    m['group'] = m.title.str.extract(r'MSC[_-](WT|SEN|G1|G2)')[0]
    m = m[m.group.notna()]
    meta = m[['title', 'group']].copy()
    meta['subject'] = m.title.str.extract(r'-(\d+)')[0].to_numpy()
    meta['unit'] = meta.subject
    meta['passage'] = np.nan
    COHORTS['GSE48662'] = dict(expr=e.loc[m.index], meta=meta, kind='log_array', tissue='BM', pooled=False, role='new', note='Control vs irradiated, and GVHD responder groups')
    LOAD_LOG.append({'cohort': 'GSE48662', 'status': 'loaded', 'samples': len(m), 'genes': e.shape[1], 'kind': 'log_array', 'tissue': 'BM', 'pooled': False, 'note': str(m.group.value_counts().to_dict())})
    print('GSE48662', e.loc[m.index].shape, m.group.value_counts().to_dict())
attempt('GSE48662', _486)
pd.DataFrame(LOAD_LOG).to_csv(RESULTS / 'ext_cohort_load_log.csv', index=False)
print(pd.DataFrame(LOAD_LOG)[['cohort', 'status', 'samples', 'genes', 'subjects', 'units', 'kind']].to_string(index=False))

## Signatures and scoring
Weights and gene lists are fixed. Rank forms are computed for all cohorts, so arrays and RNA-seq can be compared. For RNA-seq the published log1p weighted form is also computed, after the median-of-ratios step where counts allow it.

In [ ]:
pub_tbl = sg.load_sencat_markers()
PUB = pub_tbl.coef
pub_sym = pub_tbl.dropna(subset=['gene']).drop_duplicates('gene').set_index('gene').coef
REFIT = pd.read_csv(RESULTS / 'frozen_weights_refit_no_BMMSC_prAD.csv').set_index('gene_id').coef
refit_sym = REFIT.copy(); refit_sym.index = ens2sym_tbl.reindex(REFIT.index).to_numpy(); refit_sym = refit_sym[refit_sym.index.notna()]; refit_sym = refit_sym[~refit_sym.index.duplicated()]
HUSI = sg.load_husi_weights()
hall, extra = sg.load_gene_sets()
YANG7 = ['PRUNE2', 'DIO2', 'CPA4', 'PRKAA2', 'DMD', 'DDAH1', 'GATA6']
UP40 = 'CACNB4 CRYAB DAW1 EFEMP1 EPAS1 GALNT5 H1-2 KRT34 MELTF MEST MFGE8 MGARP MOXD1 SERPINB2 SERPINE2 SLC43A2 SRGN SVEP1 SYNPO2 TMEM178B'.split()
DOWN40 = 'AURKB BUB1 BUB1B CDC20 DEPDC1 KIF14 KIF18B KIF2C KIFC1 KNL1 LMNB1 NCAPH NUF2 PIMREG PKMYT1 SKA3 TNC TROAP TTK UBE2C'.split()

def rank_weighted(sym, w):
    shared = [g for g in w.index if g in sym.columns]
    wr = stats.rankdata(w.loc[shared].to_numpy())
    r = np.apply_along_axis(stats.rankdata, 1, sym[shared].fillna(0).to_numpy(dtype=float))
    wc, rc = wr - wr.mean(), r - r.mean(axis=1, keepdims=True)
    return pd.Series((rc @ wc) / (np.linalg.norm(rc, axis=1) * np.linalg.norm(wc)), index=sym.index), len(shared)

def primary_input(c):
    e = to_ens(c['expr'])
    if c['kind'] == 'counts':
        n = sg.mor_normalize(sg.author_gene_filter(e))[0]
    elif c['kind'] == 'fpkm':
        n = sg.scale_to_million(e)
    else:
        n = e
    return n

SCORES, COVER = {}, []
for key, c in COHORTS.items():
    sym = to_sym(c['expr'])
    sym = sym.fillna(sym.median()).fillna(0) if c['kind'] == 'log_array' else sym.fillna(0)
    c['sym'] = sym
    out = {}
    if c['kind'] != 'log_array':
        n = primary_input(c)
        c['norm'] = n
        out['SenCat published'], u1 = sg.weighted_log1p_score(n, PUB)
        out['SenCat refit, lineage excluded'], _ = sg.weighted_log1p_score(n, REFIT)
    else:
        out['SenCat published'], u1 = rank_weighted(sym, pub_sym)
        out['SenCat refit, lineage excluded'], _ = rank_weighted(sym, refit_sym)
    out['SenCat published, rank form'], u2 = rank_weighted(sym, pub_sym)
    out['hUSI'], u3 = sg.husi_raw(sym, HUSI)
    ranks = sym.rank(axis=1, pct=True)
    for nm, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Yang 2023 7 genes', YANG7, 1), ('Hallmark E2F (inverted)', hall['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', hall['HALLMARK_G2M_CHECKPOINT'], -1)]:
        have = [g for g in genes if g in ranks.columns]
        out[nm] = sign * ranks[have].mean(axis=1)
        COVER.append({'cohort': key, 'score': nm, 'genes_present': len(have), 'genes_total': len(genes)})
    hu, hd = [g for g in UP40 if g in ranks.columns], [g for g in DOWN40 if g in ranks.columns]
    out['AD-MSC 40 biomarkers'] = ranks[hu].mean(axis=1) - ranks[hd].mean(axis=1)
    for g, sign in [('CDKN2A', 1), ('CDKN1A', 1), ('IL6', 1), ('GDF15', 1), ('LMNB1', -1)]:
        if g in ranks.columns:
            out[g + (' (inverted)' if sign < 0 else '')] = sign * ranks[g]
    COVER.append({'cohort': key, 'score': 'SenCat published', 'genes_present': int(u1), 'genes_total': len(PUB)})
    COVER.append({'cohort': key, 'score': 'hUSI', 'genes_present': int(u3), 'genes_total': len(HUSI)})
    SCORES[key] = pd.DataFrame(out)
pd.DataFrame(COVER).to_csv(RESULTS / 'ext_signature_coverage.csv', index=False)
cov = pd.DataFrame(COVER).assign(pct=lambda d: (100 * d.genes_present / d.genes_total).round(1)).pivot(index='cohort', columns='score', values='pct')
print(cov.to_string())

husi_meta = pd.read_csv(EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')
all_text = husi_meta.astype(str).agg(' '.join, axis=1)
husi_overlap = {k.split('_')[0]: int(all_text.str.contains(k.split('_')[0]).sum()) for k in COHORTS}
print('hUSI training rows by cohort accession:', husi_overlap)
CIRCULAR = {'hUSI': {k for k, v in husi_overlap.items() if v > 0}, 'AD-MSC 40 biomarkers': {ASC}, 'Yang 2023 7 genes': {BM}}
json.dump({'husi_rows_by_accession': husi_overlap, 'circular': {k: sorted(v) for k, v in CIRCULAR.items()}}, open(RESULTS / 'ext_circularity.json', 'w'), indent=1)

long = []
for key, s in SCORES.items():
    m = COHORTS[key]['meta']
    x = s.join(m[['subject', 'unit', 'passage']]).rename_axis('sample').reset_index().melt(['sample', 'subject', 'unit', 'passage'], var_name='score_name', value_name='score')
    x.insert(0, 'cohort', key)
    long.append(x)
pd.concat(long).to_csv(RESULTS / 'ext_scores_long.csv', index=False)

## Sample identity check, independent of every score [measured]
Samples from one donor must show one sex-linked pattern. RPS4Y1 is read on log1p of the deposited scale, in RNA-seq cohorts. A donor is inconsistent when one of its samples is clearly Y-positive (above 2.5) and another clearly Y-negative (below 1.0). Within-donor trends can't be interpreted for a cohort that fails.

In [ ]:
sex_rows = []
for key, c in COHORTS.items():
    if c['kind'] == 'log_array' or 'RPS4Y1' not in c['sym'].columns or key == 'GSE48662':
        continue
    y = np.log1p(c['sym']['RPS4Y1'])
    for u, g in y.groupby(c['meta'].unit):
        sex_rows.append({'cohort': key, 'unit': u, 'samples': len(g), 'y_min': g.min(), 'y_max': g.max(), 'inconsistent': bool(g.max() > 2.5 and g.min() < 1.0)})
sex = pd.DataFrame(sex_rows)
sex.to_csv(RESULTS / 'ext_sex_integrity.csv', index=False)
summary_sex = sex.groupby('cohort').agg(units=('unit', 'size'), inconsistent=('inconsistent', 'sum'), lowest_max_y=('y_max', 'min'), highest_min_y=('y_min', 'max'))
print(summary_sex.round(2).to_string())
INTEGRITY_FAIL = sorted(summary_sex[summary_sex.inconsistent > 0].index)
print('Cohorts failing the identity check:', INTEGRITY_FAIL)
print(sex[sex.inconsistent].round(2).to_string(index=False))

## Primary result and head-to-head [measured]
Replicates at the same subject and passage are averaged. A subject's slope is per passage. Units are donors: where a donor appears in two media, its slopes are averaged first.

In [ ]:
def unit_slopes(key, name):
    c = COHORTS[key]; s = SCORES[key]
    if name not in s.columns:
        return pd.DataFrame()
    d = pd.DataFrame({'score': s[name], 'subject': c['meta'].subject, 'unit': c['meta'].unit, 'passage': c['meta'].passage}).dropna()
    d = d.groupby(['subject', 'unit', 'passage'], as_index=False).score.mean()
    rows = []
    for (sub, unit), g in d.groupby(['subject', 'unit']):
        if g.passage.nunique() < 2:
            continue
        g = g.sort_values('passage')
        rows.append({'cohort': key, 'subject': sub, 'unit': unit, 'n_passages': g.passage.nunique(), 'slope': sg.slope_per_passage(g.passage, g.score), 'rho': stats.spearmanr(g.passage, g.score).statistic if g.passage.nunique() >= 3 else np.nan,
                     'std_slope': sg.slope_per_passage(g.passage, g.score) / s[name].std()})
    t = pd.DataFrame(rows)
    return t.groupby(['cohort', 'unit'], as_index=False).agg(slope=('slope', 'mean'), rho=('rho', 'mean'), std_slope=('std_slope', 'mean'), n_passages=('n_passages', 'max')) if len(t) else t

NEW_POOLED = [k for k, c in COHORTS.items() if c['pooled']]
print('New pooled cohorts:', NEW_POOLED)
def pooled(name, cohorts, exclude=()):
    frames = [unit_slopes(k, name) for k in cohorts if k not in exclude]
    t = pd.concat([f for f in frames if len(f)]) if any(len(f) for f in frames) else pd.DataFrame()
    if t.empty:
        return None, t
    up = int((t.slope > 0).sum()); n = len(t)
    ci = stats.binomtest(up, n).proportion_ci(method='exact')
    by = t.groupby('cohort').std_slope.mean()
    return {'score': name, 'donors_rising': up, 'donors': n, 'share': up / n, 'ci_low': ci.low, 'ci_high': ci.high, 'binom_p': stats.binomtest(up, n).pvalue, 'median_rho': t.rho.median(), 'cohorts_positive': int((by > 0).sum()), 'cohorts': len(by), 'cohort_signflip_p': sg.exact_sign_flip_p(by.to_numpy())}, t

prim, prim_units = pooled('SenCat published', NEW_POOLED)
prim_units.to_csv(RESULTS / 'ext_primary_units.csv', index=False)
print('PRIMARY:', json.dumps(prim, indent=1, default=float))
print(prim_units.round(3).to_string(index=False))
fail_new = tuple(k for k in NEW_POOLED if k in INTEGRITY_FAIL)
prim_ok, prim_ok_units = pooled('SenCat published', NEW_POOLED, fail_new)
prim_ok_units.to_csv(RESULTS / 'ext_primary_units_integrity_passed.csv', index=False)
print('PRIMARY, cohorts passing the identity check:', json.dumps(prim_ok, indent=1, default=float))
sens = []
for label, name, ex in [('As registered, all pooled cohorts', 'SenCat published', ()), ('Passing identity check', 'SenCat published', fail_new), ('Passing identity check, refit without BMMSC and preadipocytes', 'SenCat refit, lineage excluded', fail_new), ('Passing identity check, rank form', 'SenCat published, rank form', fail_new), ('Passing identity check, RNA-seq only', 'SenCat published', fail_new + tuple(k for k in NEW_POOLED if COHORTS[k]['kind'] == 'log_array')), ('Rank form everywhere', 'SenCat published, rank form', ()), ('Refit without BMMSC and preadipocytes', 'SenCat refit, lineage excluded', ()), ('Excluding GSE56362', 'SenCat published', ('GSE56362',)), ('Excluding GSE233634', 'SenCat published', ('GSE233634',)), ('RNA-seq cohorts only', 'SenCat published', tuple(k for k in NEW_POOLED if COHORTS[k]['kind'] == 'log_array'))]:
    r, _ = pooled(name, NEW_POOLED, ex)
    if r:
        sens.append({'analysis': label, **{k: r[k] for k in ['donors_rising', 'donors', 'binom_p', 'median_rho', 'cohorts_positive', 'cohorts', 'cohort_signflip_p']}})
sens = pd.DataFrame(sens)
sens.to_csv(RESULTS / 'ext_primary_sensitivity.csv', index=False)
print(sens.round(4).to_string(index=False))

H2H = ['SenCat published, rank form', 'hUSI', 'SenMayo', 'Yang 2023 7 genes', 'AD-MSC 40 biomarkers', 'Hallmark E2F (inverted)', 'Hallmark G2M (inverted)', 'CDKN2A', 'CDKN1A', 'IL6', 'GDF15', 'LMNB1 (inverted)']
ALL_MULTI = [k for k in COHORTS if k not in ('GSE48662', 'GSE269586', 'GSE125632_iMSC') and k not in INTEGRITY_FAIL]
MULTI_REGISTERED = [k for k in COHORTS if k not in ('GSE48662', 'GSE269586', 'GSE125632_iMSC')]
rows = []
per_cohort = []
for name in H2H:
    ex = tuple(CIRCULAR.get(name, ()))
    r, t = pooled(name, ALL_MULTI, ex)
    if r:
        r['excluded_as_circular'] = ', '.join(sorted(ex)) or '-'
        rows.append(r)
        g = t.groupby('cohort').agg(rising=('slope', lambda v: int((v > 0).sum())), n=('slope', 'size'), median_rho=('rho', 'median')).reset_index().assign(score=name)
        per_cohort.append(g)
h2h = pd.DataFrame(rows).sort_values(['share', 'median_rho'], ascending=False)
reg = [pooled(n, MULTI_REGISTERED, tuple(CIRCULAR.get(n, ())))[0] for n in H2H]
pd.DataFrame([r for r in reg if r]).to_csv(RESULTS / 'ext_head_to_head_as_registered.csv', index=False)
h2h.to_csv(RESULTS / 'ext_head_to_head.csv', index=False)
pcdf = pd.concat(per_cohort)
pcdf.to_csv(RESULTS / 'ext_head_to_head_by_cohort.csv', index=False)
print(h2h.round(4).drop(columns=['cohort_signflip_p']).to_string(index=False))
print(pcdf.assign(x=lambda d: d.rising.astype(str) + '/' + d.n.astype(str)).pivot(index='score', columns='cohort', values='x').reindex(H2H).to_string())

## Is it specific? Random signatures, per cohort [measured]
Each cohort's own passage effect is compared with random expressed gene sets of the same size carrying the same weights.

In [ ]:
null_rows = []
for key in ALL_MULTI + ['GSE269586']:
    c = COHORTS[key]
    m = c['meta']
    logx = c['sym'].copy() if c['kind'] == 'log_array' else np.log1p(to_sym(c['norm']).fillna(0))
    d = logx.copy()
    d['subject'], d['passage'] = m.subject, m.passage
    agg = d.groupby(['subject', 'passage']).mean(numeric_only=True).reset_index()
    counts = agg.groupby('subject').passage.nunique()
    agg = agg[agg.subject.isin(counts[counts >= 2].index)]
    if agg.subject.nunique() == 0:
        continue
    X = agg.drop(columns=['subject', 'passage'])
    universe = X.columns[X.mean() > X.mean().median()]
    obs, null, p = sg.competitive_null(X, pub_sym, agg.passage.to_numpy(), agg.subject.to_numpy(), n=500, mode='genes', universe=universe)
    null_rows.append({'cohort': key, 'observed_mean_slope': obs, 'null_mean': null.mean(), 'null_sd': null.std(), 'z': (obs - null.mean()) / null.std(), 'empirical_p': p})
nulls = pd.DataFrame(null_rows)
nulls.to_csv(RESULTS / 'ext_random_signature_by_cohort.csv', index=False)
print(nulls.round(4).to_string(index=False))

## Labelled MSC checks and the clinical-response question [measured]

In [ ]:
lab_rows = []
if 'GSE48662' in SCORES:
    s, m = SCORES['GSE48662'], COHORTS['GSE48662']['meta']
    print(s.join(m.group).groupby('group').mean(numeric_only=True).T.round(3).to_string())
    for name in s.columns:
        a, b = s[name][m.group == 'WT'], s[name][m.group == 'SEN']
        lab_rows.append({'cohort': 'GSE48662', 'contrast': 'irradiated vs control', 'score': name, 'n_pos': len(b), 'n_neg': len(a), 'auroc': sg.roc_auc_score([0] * len(a) + [1] * len(b), list(a) + list(b))})
    from itertools import combinations
    g1, g2 = s[m.group == 'G1'], s[m.group == 'G2']
    resp = []
    for name in s.columns:
        vals = pd.concat([g1[name], g2[name]]).to_numpy()
        obs = g2[name].mean() - g1[name].mean()
        idx = range(len(vals))
        perm = [vals[list(c)].mean() - np.delete(vals, list(c)).mean() for c in combinations(idx, len(g2))]
        p = float(np.mean(np.array(perm) >= obs - 1e-12))
        resp.append({'score': name, 'G2_minus_G1': obs, 'one_sided_exact_p': p, 'auroc_G2_higher': sg.roc_auc_score([0] * len(g1) + [1] * len(g2), list(g1[name]) + list(g2[name]))})
    resp = pd.DataFrame(resp)
    resp.to_csv(RESULTS / 'ext_gvhd_response.csv', index=False)
    print('\nGVHD non-responders vs responders. Prespecified hypothesis is the SenCat row only; others are descriptive:')
    print(resp.round(3).to_string(index=False))
    print(s[['SenCat published']].join(m[['title', 'group']]).sort_values('SenCat published').round(4).to_string())
if 'GSE125632' in SCORES:
    s, m = SCORES['GSE125632'], COHORTS['GSE125632']['meta']
    for name in s.columns:
        a, b = s[name][m.passage == 2], s[name][m.passage == 11]
        lab_rows.append({'cohort': 'GSE125632', 'contrast': 'P11 labelled senescent vs P2', 'score': name, 'n_pos': len(b), 'n_neg': len(a), 'auroc': sg.roc_auc_score([0] * len(a) + [1] * len(b), list(a) + list(b))})
lab = pd.DataFrame(lab_rows)
lab.to_csv(RESULTS / 'ext_labelled_checks.csv', index=False)
if len(lab):
    print(lab.pivot(index='score', columns='cohort', values='auroc').round(3).to_string())

if 'GSE119987' in SCORES:
    c = COHORTS['GSE119987']; s = SCORES['GSE119987']['SenCat published']; m = c['meta']
    d = pd.DataFrame({'s': s, 'donor': m.subject, 'p': m.passage, 'pdl': m.pdl}).dropna(subset=['s'])
    rows = []
    for dn, g in d.groupby('donor'):
        g = g.groupby('p', as_index=False).mean(numeric_only=True).sort_values('p')
        base = g[g.p <= 3].s
        thr = base.mean() + 3 * max(base.std(ddof=1), 1e-9) if len(base) >= 2 else np.nan
        above = g[(g.p > 3) & (g.s > thr)]
        rows.append({'donor': dn, 'last_passage': int(g.p.max()), 'first_passage_above_early_range': int(above.p.min()) if len(above) else None, 'rho_passage': stats.spearmanr(g.p, g.s).statistic, 'rho_doubling_level': stats.spearmanr(g.pdl, g.s, nan_policy='omit').statistic})
    onset = pd.DataFrame(rows)
    onset.to_csv(RESULTS / 'ext_gse119987_onset.csv', index=False)
    print('\nGSE119987. Published work reports transcriptome change beginning after P5:')
    print(onset.round(3).to_string(index=False))

if 'GSE233634' in COHORTS and 'bioreactor' in COHORTS['GSE233634']:
    br = COHORTS['GSE233634']['bioreactor']
    bs, _ = sg.weighted_log1p_score(sg.scale_to_million(to_ens(br)), PUB)
    print('\nGSE233634 mean score: plane by passage', SCORES['GSE233634']['SenCat published'].groupby(COHORTS['GSE233634']['meta'].passage).mean().round(2).to_dict(), '| bioreactor', bs.groupby(bs.index.str[:5]).mean().round(2).to_dict())

if 'GSE269586' in SCORES:
    t = unit_slopes('GSE269586', 'SenCat published')
    print('\nGSE269586 GMP short expansion: donors rising', int((t.slope > 0).sum()), 'of', len(t), '| mean slope per passage', round(t.slope.mean(), 3))
    t.to_csv(RESULTS / 'ext_gse269586_units.csv', index=False)
if 'GSE125632_iMSC' in SCORES:
    t = unit_slopes('GSE125632_iMSC', 'SenCat published')
    print('GSE125632 iMSC P5 to P11: donors rising', int((t.slope > 0).sum()), 'of', len(t))

## Robustness: setting guard thresholds from measurement [measured]
Thinning simulates shallower sequencing; dropout simulates missing genes. Retained slope is compared with the full-data slope, per donor. Test-retest uses resequenced samples and technical replicates that already exist in the public data.

In [ ]:
rng = np.random.default_rng(sg.SEED)
def trend_mean(score, m):
    return sg.per_donor_trend(score, m.passage, m.subject)
robust = []
for key in [k for k in ['GSE137186', 'GSE275980'] if k in COHORTS]:
    raw = to_ens(COHORTS[key]['expr']).fillna(0); m = COHORTS[key]['meta']
    base_n = sg.mor_normalize(sg.author_gene_filter(raw))[0]
    base_s = sg.weighted_log1p_score(base_n, PUB)[0]
    base_slope = trend_mean(base_s, m).set_index('donor').slope_per_passage
    print(key, 'median reads per sample', int(raw.sum(axis=1).median()))
    x = raw.to_numpy(dtype=np.int64)
    for frac in [0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005]:
        for rep in range(10):
            th = pd.DataFrame(rng.binomial(x, frac), index=raw.index, columns=raw.columns)
            sc = sg.weighted_log1p_score(sg.mor_normalize(sg.author_gene_filter(th))[0], PUB)[0]
            sl = trend_mean(sc, m).set_index('donor').slope_per_passage
            robust.append({'cohort': key, 'test': 'depth', 'level': frac, 'median_reads': float(np.median(th.sum(axis=1))), 'rep': rep, 'donors_rising': int((sl > 0).sum()), 'donors': len(sl), 'slope_ratio': float((sl / base_slope).median()), 'r_with_full': float(np.corrcoef(sc, base_s)[0, 1])})
    weights = PUB[[g for g in PUB.index if g in base_n.columns]]
    for frac in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]:
        for rep in range(20):
            keep = rng.random(len(weights)) > frac
            w = weights[keep]
            sc = sg.weighted_log1p_score(base_n, w)[0]
            sl = trend_mean(sc, m).set_index('donor').slope_per_passage
            robust.append({'cohort': key, 'test': 'dropout', 'level': frac, 'median_reads': np.nan, 'rep': rep, 'donors_rising': int((sl > 0).sum()), 'donors': len(sl), 'slope_ratio': float((sl / base_slope).median()), 'r_with_full': float(np.corrcoef(sc, base_s)[0, 1])})
    # A lab has few lots, so each sample is removed in turn to see how much the within-cohort reference moves other samples' scores
    loo = []
    for s in raw.index:
        sub = raw.drop(index=s)
        sc = sg.weighted_log1p_score(sg.mor_normalize(sg.author_gene_filter(sub))[0], PUB)[0]
        loo.append(float((sc - base_s.reindex(sc.index)).abs().max()))
    print(key, 'leave-one-sample-out: largest change in any other sample score', round(max(loo), 3), '| score SD', round(base_s.std(), 2))
    pd.DataFrame({'sample_removed': raw.index, 'max_abs_change': loo}).to_csv(RESULTS / f'ext_robust_leave_one_out_{key}.csv', index=False)
    retest = []
    d = pd.DataFrame({'s': base_s, 'subj': m.subject, 'p': m.passage})
    for (sb, p), g in d.groupby(['subj', 'p']):
        if len(g) > 1:
            retest.append({'cohort': key, 'subject': sb, 'passage': p, 'n': len(g), 'difference': float(g.s.max() - g.s.min())})
    per_step = trend_mean(base_s[~base_s.index.duplicated()], m).slope_per_passage.abs().median()
    if retest:
        rt = pd.DataFrame(retest)
        rt['slope_per_passage_median'] = per_step
        rt.to_csv(RESULTS / 'ext_test_retest_seq.csv', index=False)
        print(rt.round(3).to_string(index=False))
rob = pd.DataFrame(robust)
rob.to_csv(RESULTS / 'ext_robustness.csv', index=False)
if len(rob):
    summ = rob.groupby(['cohort', 'test', 'level']).agg(median_reads=('median_reads', 'median'), share_donors_rising=('donors_rising', lambda v: v.sum() / rob.loc[v.index, 'donors'].sum()), slope_ratio=('slope_ratio', 'median'), r_with_full=('r_with_full', 'median')).reset_index()
    summ.to_csv(RESULTS / 'ext_robustness_summary.csv', index=False)
    print(summ.round(3).to_string(index=False))

if 'GSE56362' in SCORES:
    s, m = SCORES['GSE56362']['SenCat published'], COHORTS['GSE56362']['meta']
    d = pd.DataFrame({'s': s, 'subj': m.subject, 'p': m.passage})
    tech_sd = d.groupby(['subj', 'p']).s.std().median()
    steps = d.groupby(['subj', 'p']).s.mean().groupby('subj').apply(lambda g: g.max() - g.min()).median()
    print(f'\nGSE56362: technical SD {tech_sd:.4f}; median donor range across passages {steps:.4f}; ratio {steps / tech_sd:.1f}')
    json.dump({'technical_sd': float(tech_sd), 'median_range': float(steps)}, open(RESULTS / 'ext_test_retest_array.json', 'w'))

if 'GSE194298' in SCORES:
    s, m = SCORES['GSE194298']['SenCat published'], COHORTS['GSE194298']['meta']
    d = pd.DataFrame({'s': s, 'donor': m.donor, 'medium': m.medium, 'p': m.passage})
    print('\nGSE194298 mean score by medium and passage:')
    print(d.groupby(['medium', 'p']).s.mean().unstack().round(2).to_string())
    w = d.pivot_table(index=['donor', 'p'], columns='medium', values='s')
    print('Medium effect at the same donor and passage (hPL 0/AB minus FBS):', (w['HPL 0/AB'] - w['FBS']).groupby('p').mean().round(2).to_dict())
    d.to_csv(RESULTS / 'ext_medium_effect.csv', index=False)

early = []
for key in [k for k in SCORES if COHORTS[k]['kind'] in ('counts', 'normalized', 'normcounts') and k != 'GSE269586']:
    s = SCORES[key]['SenCat published']; m = COHORTS[key]['meta']
    g = pd.DataFrame({'s': s, 'p': m.passage}).dropna()
    g = g[g.p == g.p.min()]
    early.append({'cohort': key, 'earliest_passage': g.p.iloc[0], 'n': len(g), 'mean_score': g.s.mean(), 'sd': g.s.std()})
early = pd.DataFrame(early)
early.to_csv(RESULTS / 'ext_absolute_scale.csv', index=False)
print('\nAbsolute score at the earliest passage, count-based cohorts:')
print(early.round(2).to_string(index=False))

## Drift clock: a trained model under leave-one-cohort-out [measured]
Input is each sample's gene ranks over genes shared by all cohorts. Within-donor centering removes donor and lab level. Each held-out cohort is scored by a model fit only on the others, with ridge strength chosen by inner leave-one-cohort-out. The negative control trains on passages shuffled within donors.

In [ ]:
from sklearn.linear_model import Ridge
CLOCK_COHORTS = [k for k in ALL_MULTI if k in COHORTS]   # the identity-check failure cannot supply within-donor targets
shared = None
for k in CLOCK_COHORTS:
    cols = set(COHORTS[k]['sym'].columns)
    shared = cols if shared is None else shared & cols
shared = sorted(shared)
print('Clock cohorts', CLOCK_COHORTS, '| shared genes', len(shared))
frames = []
for k in CLOCK_COHORTS:
    c = COHORTS[k]
    R = c['sym'][shared].rank(axis=1, pct=True)
    R = R.join(c['meta'][['subject', 'unit', 'passage']])
    a = R.groupby(['subject', 'unit', 'passage'], as_index=False).mean(numeric_only=True)
    a = a[a.groupby('subject').passage.transform('nunique') >= 2]
    a.insert(0, 'cohort', k)
    frames.append(a)
D = pd.concat(frames, ignore_index=True)
Xc = D[shared].to_numpy(dtype=np.float32)
y0 = np.log2(D.passage.to_numpy() + 1)
subj = (D.cohort + '|' + D.subject).to_numpy()
def center(v):
    return v - pd.DataFrame(v).groupby(subj).transform('mean').to_numpy()
Xw = center(Xc)
yw = center(y0.reshape(-1, 1)).ravel()
ALPHAS = [0.1, 1, 10, 100, 1000]
def fit(train, y, alpha):
    sd = Xw[train].std(axis=0) + 1e-6
    return Ridge(alpha=alpha).fit(Xw[train] / sd, y[train]), sd
def choose(train, y):
    best, score = ALPHAS[0], np.inf
    groups = D.cohort.to_numpy()[train]
    for a in ALPHAS:
        err = []
        for g in np.unique(groups):
            tr, te = train[groups != g], train[groups == g]
            m, sd = fit(tr, y, a)
            err.append(np.mean((m.predict(Xw[te] / sd) - y[te]) ** 2))
        if np.mean(err) < score:
            best, score = a, float(np.mean(err))
    return best
pred = np.full(len(D), np.nan)
clock_rows, alpha_used, LOCO_W = [], {}, {}
cohorts = D.cohort.to_numpy()
for k in CLOCK_COHORTS:
    tr, te = np.flatnonzero(cohorts != k), np.flatnonzero(cohorts == k)
    a = choose(tr, yw)
    m, sd = fit(tr, yw, a)
    pred[te] = m.predict(Xw[te] / sd)
    alpha_used[k] = a
    LOCO_W[k] = (m.coef_ / sd).tolist()
D['clock_pred'] = pred
rng2 = np.random.default_rng(sg.SEED)
def shuffled_y():
    y = y0.copy()
    for s in np.unique(subj):
        i = np.flatnonzero(subj == s)
        y[i] = rng2.permutation(y[i])
    return center(y.reshape(-1, 1)).ravel()
neg_rhos = []
for rep in range(10):
    ys = shuffled_y()
    pn = np.full(len(D), np.nan)
    for k in CLOCK_COHORTS:
        tr, te = np.flatnonzero(cohorts != k), np.flatnonzero(cohorts == k)
        m, sd = fit(tr, ys, alpha_used[k])
        pn[te] = m.predict(Xw[te] / sd)
    d = D.assign(pn=pn)
    rs = [stats.spearmanr(g.passage, g.pn).statistic for _, g in d.groupby(['cohort', 'subject']) if g.passage.nunique() >= 3]
    neg_rhos.append(float(np.nanmedian(rs)))
for k, g in D.groupby('cohort'):
    for (sb, u), h in g.groupby(['subject', 'unit']):
        h = h.sort_values('passage')
        clock_rows.append({'cohort': k, 'subject': sb, 'unit': u, 'n_passages': h.passage.nunique(), 'slope': sg.slope_per_passage(h.passage, h.clock_pred), 'rho': stats.spearmanr(h.passage, h.clock_pred).statistic if h.passage.nunique() >= 3 else np.nan,
                           'error_log2': float(np.mean(np.abs(h.clock_pred - (np.log2(h.passage + 1) - np.log2(h.passage + 1).mean()))))})
cl = pd.DataFrame(clock_rows)
cl.to_csv(RESULTS / 'ext_clock_loco.csv', index=False)
by = cl.groupby('cohort').agg(rising=('slope', lambda v: int((v > 0).sum())), n=('slope', 'size'), median_rho=('rho', 'median'), mae_log2=('error_log2', 'median')).assign(alpha=pd.Series(alpha_used))
print(by.round(3).to_string())
sc_cmp = pd.concat([unit_slopes(k, 'SenCat published, rank form').assign(score='SenCat rank') for k in CLOCK_COHORTS])
gate = {'median_rho': float(cl.rho.median()), 'share_rising': float((cl.slope > 0).mean()), 'shuffled_median_rho': float(np.median(neg_rhos)), 'shuffled_runs': neg_rhos, 'sencat_rank_median_rho': float(sc_cmp.rho.median()), 'sencat_rank_share_rising': float((sc_cmp.slope > 0).mean())}
gate['passes'] = bool(gate['median_rho'] >= 0.8 and gate['share_rising'] >= 0.9 and gate['shuffled_median_rho'] < 0.3)
print(json.dumps(gate, indent=1))
a_all = choose(np.arange(len(D)), yw)
sd_all = Xw.std(axis=0) + 1e-6
final = Ridge(alpha=a_all).fit(Xw / sd_all, yw)
pd.DataFrame({'gene': shared, 'coef_per_rank_unit': final.coef_ / sd_all}).to_csv(RESULTS / 'drift_clock_weights.csv', index=False)
gate['final_alpha'] = a_all
gate['shared_genes'] = len(shared)
json.dump(gate, open(RESULTS / 'ext_clock_gate.json', 'w'), indent=1)
top = pd.DataFrame({'gene': shared, 'c': final.coef_ / sd_all}).sort_values('c')
print('Most negative:', top.gene.head(15).tolist()); print('Most positive:', top.gene.tail(15).tolist())
in_e2f = set(hall['HALLMARK_E2F_TARGETS']) | set(hall['HALLMARK_G2M_CHECKPOINT'])
print('Share of top 100 negative clock genes in E2F/G2M:', round(top.gene.head(100).isin(in_e2f).mean(), 3))
D[['cohort', 'subject', 'unit', 'passage', 'clock_pred']].to_csv(RESULTS / 'ext_clock_predictions.csv', index=False)
sym_to_ens = {}
for e, sy in ens2sym_tbl.items():
    if sy in set(shared):
        sym_to_ens.setdefault(sy, []).append(e)
json.dump({'genes': shared, 'ensembl_by_symbol': sym_to_ens, 'models': {'final': (final.coef_ / sd_all).tolist(), **{f'without_{k}': LOCO_W[k] for k in LOCO_W}}, 'alpha': {'final': a_all, **alpha_used}, 'seed': sg.SEED}, open(RESULTS / 'drift_clock_models.json', 'w'))

## Figures and report

In [ ]:
order = h2h.sort_values('share').score.tolist()
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={'width_ratios': [1.1, 1]})
ax = axes[0]
for i, n in enumerate(order):
    r = h2h[h2h.score == n].iloc[0]
    ax.plot([r.ci_low, r.ci_high], [i, i], color=sg.PAL['blue'], lw=2, solid_capstyle='round')
    ax.scatter([r.share], [i], s=44, color=sg.PAL['blue'], edgecolors=sg.PAL['surface'], linewidths=1.5, zorder=3)
    ax.text(1.02, i, f'{int(r.donors_rising)}/{int(r.donors)}', va='center', fontsize=8.5, color=sg.PAL['ink2'])
ax.axvline(0.5, color=sg.PAL['muted'], lw=1, ls=(0, (3, 3)))
ax.set_yticks(range(len(order)), order, fontsize=8.5); ax.set_xlim(0, 1.12); ax.set_xlabel('Share of donors with a rising score, exact 95% CI'); ax.set_title('A. Head-to-head across MSC cohorts'); ax.grid(axis='y', visible=False)
ax = axes[1]
if len(rob):
    a = summ[(summ.test == 'depth') & (summ.cohort == 'GSE137186')].sort_values('median_reads')
    ax.plot(a.median_reads / 1e6, a.share_donors_rising, color=sg.PAL['blue'], marker='o', ms=6, mec=sg.PAL['surface'], mew=1.5, label='Donors still rising')
    ax.plot(a.median_reads / 1e6, a.slope_ratio, color=sg.PAL['orange'], marker='o', ms=6, mec=sg.PAL['surface'], mew=1.5, label='Slope, relative to full depth')
    ax.set_xscale('log'); ax.set_xlabel('Median million reads per sample, thinned'); ax.set_ylabel('Value'); ax.set_title('B. Sequencing depth'); ax.legend(fontsize=8, loc='lower right'); ax.grid(axis='x', visible=False)
fig.tight_layout()
fig.savefig(sg.FIGS / 'fig6_head_to_head_depth.png')
plt.show()

show = [k for k in ['GSE137186', 'GSE275980', 'GSE125632', 'GSE194298', 'GSE119987', 'GSE56362', 'GSE233634'] if k in COHORTS]
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 3.1))
    for ax, k in zip(np.atleast_1d(axes), show):
        s = SCORES[k]['SenCat published']; m = COHORTS[k]['meta']
        d = pd.DataFrame({'s': s, 'subject': m.subject, 'p': m.passage}).groupby(['subject', 'p'], as_index=False).s.mean()
        for _, g in d.groupby('subject'):
            g = g.sort_values('p')
            ax.plot(g.p, g.s, color=sg.PAL['blue'], lw=1.5, marker='o', ms=4, mec=sg.PAL['surface'], mew=1)
        ax.set_title(k, fontsize=9.5); ax.set_xlabel('Passage'); ax.grid(axis='x', visible=False)
    np.atleast_1d(axes)[0].set_ylabel('SenCat score')
    fig.tight_layout()
    fig.savefig(sg.FIGS / 'fig7_new_cohorts.png')
    plt.show()

deviations = f"""# Deviations from amendment 1

The first run of this analysis (kernel E version 1) exposed four problems. Each was fixed and everything was rerun. Results of the first run stay in kaggle_runs/e_v1.

1. **GSE48662 G2 samples were not parsed.** Their titles use a hyphen (MSC-G2-1). The first run's response table was empty and meaningless.
2. **GSE56362 failed to load.** Its platform labels symbols GENE_NAME. Fixed, so this registered cohort is now included.
3. **GSE119987 counted F2 twice.** F2.2 is a second culture of donor F2, and the series states 4 donors. First run: 16 of 22 donors rising.
4. **GSE125632 failed a sample identity check that does not depend on any score.** GEO lists donor 3 as the only male, but Y-linked expression sits in samples 5, 6, 13, 15 and 16, so its donor pairing is contradicted. The check itself was not registered in advance; it came from a planned column-order check that returned 65% agreement. This cohort is therefore reported both ways, and its within-donor units are not used in the head-to-head or the clock. In the first run the clock failed its gate with this cohort included (88% of held-out donors rising against the 90% required).

Results below come after these fixes. The scores, weights and tests are unchanged.
"""
(REPORTS / 'AMENDMENT1_DEVIATIONS.md').write_text(deviations, encoding='utf-8')

report = f"""# Amendment 1: extended MSC evaluation

Amendment SHA256: {sg.sha256(amend_path)}. Cohort load log: results/ext_cohort_load_log.csv.

## Primary as registered, with parsing fixes [measured]
{json.dumps(prim, indent=1, default=float)}

## Primary, cohorts passing the identity check [measured]
{json.dumps(prim_ok, indent=1, default=float)}

## Identity check [measured]
{summary_sex.round(2).to_string()}

## Sensitivity [measured]
{sens.round(4).to_string(index=False)}

## Head-to-head [measured]
{h2h.round(4).to_string(index=False)}

## Random-signature comparison by cohort [measured]
{nulls.round(4).to_string(index=False)}

## Labelled checks [measured]
{lab.round(3).to_string(index=False) if len(lab) else 'Not available.'}

## GVHD response [measured, hypothesis-generating]
{resp.round(3).to_string(index=False) if 'resp' in dir() else 'Not available.'}

## Drift clock [measured]
{by.round(3).to_string()}

Gate: {json.dumps(gate)}

## Robustness [measured]
{summ.round(3).to_string(index=False) if len(rob) else 'Not available.'}
"""
(REPORTS / 'AMENDMENT1_RESULTS.md').write_text(report, encoding='utf-8')
print(report[:800])